In [1]:

import heapq

# Goal state
GOAL = (1, 2, 3,
        4, 5, 6,
        7, 8, 0)  # 0 represents the blank tile


# Find the position of the blank tile
def get_neighbors(state):
    neighbors = []
    zero = state.index(0)
    row, col = divmod(zero, 3)

    moves = [
        (-1, 0),  # Up
        (1, 0),   # Down
        (0, -1),  # Left
        (0, 1)    # Right
    ]

    for dr, dc in moves:
        nr, nc = row + dr, col + dc

        if 0 <= nr < 3 and 0 <= nc < 3:
            new_pos = nr * 3 + nc
            new_state = list(state)

            # Swap blank with adjacent tile
            new_state[zero], new_state[new_pos] = (
                new_state[new_pos], new_state[zero]
            )

            neighbors.append(tuple(new_state))

    return neighbors


# Heuristic 1: Number of misplaced tiles
def misplaced_tiles(state):
    return sum(
        1 for i in range(9)
        if state[i] != 0 and state[i] != GOAL[i]
    )


# Heuristic 2: Manhattan distance
def manhattan_distance(state):
    distance = 0

    for i, tile in enumerate(state):
        if tile == 0:
            continue

        current_row, current_col = divmod(i, 3)
        goal_pos = GOAL.index(tile)
        goal_row, goal_col = divmod(goal_pos, 3)

        distance += (
            abs(current_row - goal_row)
            + abs(current_col - goal_col)
        )

    return distance


# A* search algorithm
def astar(start, heuristic):
    # Priority queue: (f, g, state)
    pq = [(heuristic(start), 0, start)]

    # Best known depth for each state
    g_cost = {start: 0}

    # Store parent states to reconstruct the solution
    parent = {start: None}

    while pq:
        f, g, state = heapq.heappop(pq)

        # Ignore outdated queue entries
        if g != g_cost[state]:
            continue

        # Goal reached
        if state == GOAL:
            path = []
            current = state

            while current is not None:
                path.append(current)
                current = parent[current]

            return path[::-1]

        # Explore neighboring states
        for neighbor in get_neighbors(state):
            new_g = g + 1

            if new_g < g_cost.get(neighbor, float("inf")):
                g_cost[neighbor] = new_g
                parent[neighbor] = state

                new_f = new_g + heuristic(neighbor)
                heapq.heappush(
                    pq, (new_f, new_g, neighbor)
                )

    return None


# Print the solution path
def print_solution(path):
    if path is None:
        print("No solution exists.")
        return

    print("Number of moves:", len(path) - 1)

    for step, state in enumerate(path):
        print(f"\nStep {step}:")
        for i in range(0, 9, 3):
            print(state[i:i + 3])


# Initial state (change as needed)
start = (1, 2, 3,
         4, 0, 6,
         7, 5, 8)

# Case 1: Misplaced Tiles
print("A* with Misplaced Tiles Heuristic")
path1 = astar(start, misplaced_tiles)
print_solution(path1)

# Case 2: Manhattan Distance
print("\nA* with Manhattan Distance Heuristic")
path2 = astar(start, manhattan_distance)
print_solution(path2)


A* with Misplaced Tiles Heuristic
Number of moves: 2

Step 0:
(1, 2, 3)
(4, 0, 6)
(7, 5, 8)

Step 1:
(1, 2, 3)
(4, 5, 6)
(7, 0, 8)

Step 2:
(1, 2, 3)
(4, 5, 6)
(7, 8, 0)

A* with Manhattan Distance Heuristic
Number of moves: 2

Step 0:
(1, 2, 3)
(4, 0, 6)
(7, 5, 8)

Step 1:
(1, 2, 3)
(4, 5, 6)
(7, 0, 8)

Step 2:
(1, 2, 3)
(4, 5, 6)
(7, 8, 0)
